In [ ]:
# Install dependencies
!pip install torch torchvision --index-url https://pytorch.org
!pip install diffusers transformers accelerate safetensors pillow

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive') # Access Drive folder

In [ ]:
# List everything in the drive folder
!ls -la /content/drive/MyDrive/generative_rockets/

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
from PIL import Image
import os
import gc
import random

assert torch.cuda.is_available(), "CUDA is required to run SDXL locally."

In [ ]:
# Makes sure there are not more pipelines running in background
if "pipeline" in globals():
    del pipeline

# downloads model from Hugging Face cache
pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")

# downloads the adapter subfolder weights
pipeline.load_ip_adapter(
    "h94/IP-Adapter", 
    subfolder="sdxl_models", 
    weight_name="ip-adapter_sdxl.bin"
)

In [ ]:
# Set strength scale (0.5 to 0.8 balances image influence vs text prompt)
pipeline.set_ip_adapter_scale(0.6)

# Force the VAE decoder to match the float16 pipeline precision
pipeline.vae.to(dtype=torch.float16)

# Setting up memory saving configurations
pipeline.enable_freeu(s1=0.9, s2=0.2, b1=1.3, b2=1.4)
pipeline.unet.to(memory_format=torch.channels_last) 
pipeline.vae.enable_slicing()
pipeline.vae.enable_tiling()

In [ ]:
# Forces Python's Garbage Collector to scan and destroy unlinked objects
gc.collect()

# Empty PyTorch's internal GPU memory cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

In [ ]:
image_dir = "/content/drive/MyDrive/generative_rockets/" 
image_paths = random.sample([os.path.join(image_dir, f) for f in sorted(os.listdir(image_dir)) if f.endswith(('png', 'jpg', 'jpeg'))],15)
print(f"Image paths: ", image_paths)
output_path = "generated_fusion_spaceship.png"

reference_images = []

for p in image_paths:
    img = Image.open(p)
    img_rgb = img.convert("RGB")
    reference_images.append(img_rgb)

prompt = (
    "A spaceship far away. The spaceship is human made. Make sure to show the whole spaceship"
    "combining structural design cues from the reference images, "
    "we should see the engines"
    "intricate metal armor plating, glowing purple and yellow ion engines, it has a recolector of dark martter in the front,"
    "the engines are in the back and the sides, 8k resolution"
)

print("Generating image...")

generator = torch.Generator(device="cuda").manual_seed(1720)
generated_image = pipeline(
    prompt=prompt,
    ip_adapter_image=[reference_images],
    num_inference_steps=50,
    generator=generator,
).images[0]


generated_image.save(output_path)
print(f"Saved generated spaceship to {output_path}")
